# Bentopy Tutorial 1 — Basic Packing in Empty Space

This notebook follows the [Bentopy tutorial](https://cgmartini.nl/docs/tutorials/Martini3/Bentopy/)
section *Tutorial 1: Basic Packing in Empty Space*.

Workflow:

1. Write a `.bent` recipe describing the system.
2. Run **`bentopy-pack`** to pack 650 lysozymes into a 40 × 40 × 40 nm³ box.
3. Run **`bentopy-render`** to convert the placement list into a `.gro` file
   plus a GROMACS topology.
4. Run **`bentopy-solvate`** to add water and 0.15 M NaCl.
5. Inspect the final solvated system with VMD.


## 0. Verify the workspace

Before opening this notebook you should have already run

```bash
bash /projects/bgvl/alfiaparvez/bentopy_tutorial/notebooks/setup_martini.sh
```

That script created `/projects/bgvl/$USER/Martini/tutorial_1/` with a
copy of this notebook, a copy of `structures/`, `topology/`, `mdp_files/`,
and the visualization script `vis_t1.tcl`.  When you opened *this* copy
of the notebook, JupyterLab set the kernel's working directory to that
folder, which is where every output below will land.

The next cell checks the layout and ensures `bentopy-pack` is on `$PATH`.


In [ ]:
import os, pathlib, shutil, sys

# Make the bentopy CLI tools available no matter which Python kernel the
# user picks. The shared venv lives in the workshop's allocation.
VENV_BIN = "/projects/bgvl/alfiaparvez/bentopy_tutorial/.venv/bin"
if VENV_BIN not in os.environ["PATH"]:
    os.environ["PATH"] = VENV_BIN + ":" + os.environ["PATH"]

# This notebook is meant to be opened from inside its tutorial folder,
# e.g.  /projects/bgvl/$USER/SummerSchool_2026/Martini/tutorial_1/ .  Outputs of
# every cell below (.bent, .json, .gro, .top, ...) land in this folder.
NB_DIR = pathlib.Path.cwd()

missing = [d for d in ("structures", "topology", "mdp_files")
           if not (NB_DIR / d).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing input directories in {NB_DIR}: {missing}.\n"
        f"Run  bash Martini/copy.sh  first, then open the notebook from "
        f"/projects/bgvl/$USER/SummerSchool_2026/Martini/tutorial_1/."
    )

print("Working dir :", NB_DIR)
print("bentopy-pack:", shutil.which("bentopy-pack"))


## 1. Create the packing configuration (`simple_packing.bent`)

The `.bent` file is a small recipe describing the box, the input topologies,
the compartments and which segments to pack into them.


In [ ]:
bent = '''[ general ]
title "Proteins in a box"
seed 0

[ space ]
# All dimensions in bentopy are given in nanometers.
dimensions 40, 40, 40
resolution 0.5

[ includes ]
"topology/martini_v3.0.0.itp"
"topology/martini_v3.0.0_ions_v1.itp"
"topology/martini_v3.0.0_solvents_v1.itp"
"topology/lysozyme.itp"

[ compartments ]
system is all

[ segments ]
LYZ 650 from "structures/lysozyme.pdb" in system
'''
pathlib.Path("simple_packing.bent").write_text(bent)
print(bent)


## 2. Pack the system

`bentopy-pack` reads the recipe and writes a *placement list* — a
lightweight JSON describing where each molecule was placed. It does **not**
yet write atomic coordinates.


In [ ]:
!bentopy-pack simple_packing.bent placements.json


## 3. Render the placements to a coordinate file

`bentopy-render` expands the placement list into a real `.gro` coordinate
file together with a `.top` topology suitable for GROMACS.


In [ ]:
!bentopy-render placements.json system.gro -t topol.top
print('--- topol.top ---')
print(pathlib.Path('topol.top').read_text())


## 4. Solvation

We add water plus 150 mM NaCl, and let `bentopy-solvate` compute the extra
counter-ions needed to make the system electrically neutral.


In [ ]:
!bentopy-solvate -i system.gro -o solvated_system.gro \
    -s NA:0.15M -s CL:0.15M --charge neutral -t topol.top
print('--- final topol.top ---')
print(pathlib.Path('topol.top').read_text())


## 5. Visualize

Open the solvated system with VMD (run this in a Delta OnDemand desktop
session, not in the notebook itself):

```bash
export PATH=/projects/bgvl/alfiaparvez/software/vmd/bin:$PATH
cd /projects/bgvl/$USER/SummerSchool_2026/Martini/tutorial_1
vmd -e vis_t1.tcl solvated_system.gro
```

The QuickSurf rendering script `vis_t1.tcl` is in this folder already,
so VMD will pick it up directly.


### Expected result (Figure 4 of the tutorial)

* 650 lysozymes uniformly distributed throughout the box.
* No overlaps between proteins.
* Cytoplasmic densities; the `bentopy-pack` summary should report **100.0 %**
  of the requested copies placed.
